In [3]:
from google import genai
from google.genai import types
import json
from pathlib import Path
from dotenv import load_dotenv
import os


load_dotenv() 




GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")  
MODEL_NAME = "gemini-2.5-flash"   
EPUB_PATH = Path("../uploads/red-rising/book_0.epub")  # Change this path to your EPUB file

client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client ready:", MODEL_NAME)

Gemini client ready: gemini-2.5-flash


In [5]:
import sys

from src.ingestion.epub_parser import parse_epub  

def chapters_to_dicts(parsed_chapters) -> list[dict]:
    """Adapt ParsedChapter objects to the dict format the notebook uses."""
    return [
        {
            "index":      ch.index + 1,      # 1-based for display
            "title":      ch.label,
            "text":       ch.text,
            "word_count": len(ch.text.split()),
        }
        for ch in parsed_chapters
    ]

parsed = parse_epub(EPUB_PATH)
chapters = chapters_to_dicts(parsed)

print(f"Found {len(chapters)} chapters\n")
for ch in chapters:
    print(f"  {ch['index']:>3}. {ch['title']:<40} {ch['word_count']:>6} words")

Found 50 chapters

    1. Prologue                                    269 words
    2. 1: Helldiver                               2260 words
    3. 2: The Township                            3417 words
    4. 3: The Laurel                              2801 words
    5. 4: The Gift                                2736 words
    6. 5: The First Song                          2664 words
    7. 6: The Martyr                              2390 words
    8. 7: Lazarus                                 1378 words
    9. 8: Dancer                                  2999 words
   10. 9: The Lie                                 1294 words
   11. 10: The Carver                             2932 words
   12. 11: Mad                                     915 words
   13. 12: The Carving                            4112 words
   14. 13: Bad Things                             2251 words
   15. 14: Andromedus                             1353 words
   16. 15: The Testing                            2042 words
   17

In [6]:
class KnowledgeRegistry:
    """In-memory store of everything extracted so far. 
    Gets serialized into prompts so the model can reference existing entities by ID."""

    def __init__(self):
        self.characters    = {}   # cid -> {name, brief, traits, revelations, first_chapter}
        self.locations     = {}   # lid -> {name, description, first_chapter}
        self.organizations = {}   # oid -> {name, description, first_chapter}
        self.relationships = {}   # "Ca|Cb" -> {type, developments}
        self._counters     = {"C": 0, "L": 0, "O": 0}

    def _next_id(self, prefix):
        self._counters[prefix] += 1
        return f"{prefix}{self._counters[prefix]}"

    # ── Compact serialization for prompt injection ──────────────────────────

    def characters_str(self):
        if not self.characters:
            return "None yet."
        return "\n".join(
            f"  [{cid}] {c['name']} -- {c.get('brief', 'no description')}"
            for cid, c in self.characters.items()
        )

    def locations_str(self):
        if not self.locations:
            return "None yet."
        return "\n".join(f"  [{lid}] {l['name']}" for lid, l in self.locations.items())

    # ── Apply deltas to registry ────────────────────────────────────────────

    def apply_characters(self, delta: dict, chapter_num: int):
        for char in delta.get("characters", []):
            if char.get("is_new"):
                cid = self._next_id("C")
                self.characters[cid] = {
                    "name":          char.get("name", "Unknown"),
                    "brief":         char.get("brief_description", ""),
                    "traits":        [char.get("new_traits", "")] if char.get("new_traits") else [],
                    "revelations":   {chapter_num: char["new_revelations"]} if char.get("new_revelations") else {},
                    "first_chapter": chapter_num,
                    "aliases":       char.get("aliases", []),
                }
                char["resolved_id"] = cid
                print(f"    + New character: [{cid}] {char['name']}")
            else:
                cid = char.get("id")
                if cid and cid in self.characters:
                    if char.get("new_traits"):
                        self.characters[cid]["traits"].append(char["new_traits"])
                    if char.get("new_revelations"):
                        self.characters[cid]["revelations"][chapter_num] = char["new_revelations"]
                    print(f"    ~ Updated character: [{cid}] {self.characters[cid]['name']}")

    def apply_relationships(self, delta: dict, chapter_num: int):
        for rel in delta.get("relationships", []):
            a, b = rel.get("character_a_id"), rel.get("character_b_id")
            if not a or not b:
                continue
            key = "|".join(sorted([a, b]))
            if key not in self.relationships:
                self.relationships[key] = {
                    "character_a_id": a,
                    "character_b_id": b,
                    "type":           rel.get("relationship_type", "unknown"),
                    "developments":   {},
                }
            if rel.get("development"):
                self.relationships[key]["developments"][chapter_num] = rel["development"]
            print(f"    ~ Relationship: {a} <-> {b} ({rel.get('relationship_type')})")

    def apply_world(self, delta: dict, chapter_num: int):
        for loc in delta.get("new_locations", []):
            lid = self._next_id("L")
            self.locations[lid] = {
                "name":          loc["name"],
                "description":   loc.get("description", ""),
                "first_chapter": chapter_num,
            }
            print(f"    + New location: [{lid}] {loc['name']}")
        for org in delta.get("new_organizations", []):
            oid = self._next_id("O")
            self.organizations[oid] = {
                "name":          org["name"],
                "description":   org.get("description", ""),
                "first_chapter": chapter_num,
            }
            print(f"    + New organization: [{oid}] {org['name']}")

registry = KnowledgeRegistry()
print("Registry initialized.")

Registry initialized.


In [44]:
import time

def call_gemini(prompt: str, schema: types.Schema, max_tokens: int = 4096, retries: int = 3) -> dict:
    for attempt in range(retries):
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=schema,
                temperature=0.1,
                max_output_tokens=max_tokens,
            ),
        )

        usage = response.usage_metadata
        print(f"    tokens -- input: {usage.prompt_token_count}, output: {usage.candidates_token_count}, limit: {max_tokens}")

        if response.text is None:
            candidate = response.candidates[0] if response.candidates else None
            finish_reason = candidate.finish_reason if candidate else "no candidates"
            print(f"    attempt {attempt + 1}/{retries} failed -- finish_reason: {finish_reason}")
            
            if attempt < retries - 1:
                time.sleep(2 ** attempt)  # 1s, 2s, 4s backoff
                continue
            
            raise ValueError(f"Gemini returned no text after {retries} attempts. finish_reason: {finish_reason}")

        try:
            return json.loads(response.text)
        except json.JSONDecodeError:
            print(f"RAW RESPONSE (failed to parse):\n{response.text[:800]}")
            raise

    raise ValueError("Exhausted retries without a result")

In [40]:
from google.genai import types

# ── Schemas ─────────────────────────────────────────────────────────────────

SUMMARY_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    required=["chapter_number", "chapter_title", "summary", "tone"],
    properties={
        "chapter_number":  types.Schema(type=types.Type.INTEGER),
        "chapter_title":   types.Schema(type=types.Type.STRING),
        "summary":         types.Schema(type=types.Type.STRING),
        "pov_character":   types.Schema(type=types.Type.STRING),
        "tone":            types.Schema(type=types.Type.STRING),
    },
)

CHARACTERS_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    required=["characters"],
    properties={
        "characters": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(
                type=types.Type.OBJECT,
                required=["is_new", "name", "role_in_chapter"],
                properties={
                    "is_new":            types.Schema(type=types.Type.BOOLEAN),
                    "id":                types.Schema(type=types.Type.STRING),
                    "name":              types.Schema(type=types.Type.STRING),
                    "aliases":           types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING)),
                    "brief_description": types.Schema(type=types.Type.STRING),
                    "new_traits":        types.Schema(type=types.Type.STRING),
                    "new_revelations":   types.Schema(type=types.Type.STRING),
                    "role_in_chapter":   types.Schema(type=types.Type.STRING),
                },
            ),
        )
    },
)

RELATIONSHIPS_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    required=["relationships"],
    properties={
        "relationships": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(
                type=types.Type.OBJECT,
                required=["character_a_id", "character_b_id", "relationship_type", "is_new"],
                properties={
                    "character_a_id":    types.Schema(type=types.Type.STRING),
                    "character_b_id":    types.Schema(type=types.Type.STRING),
                    "relationship_type": types.Schema(type=types.Type.STRING),
                    "is_new":            types.Schema(type=types.Type.BOOLEAN),
                    "development":       types.Schema(type=types.Type.STRING),
                },
            ),
        )
    },
)

WORLD_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    required=["new_locations", "new_organizations", "new_lore"],
    properties={
        "new_locations": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(
                type=types.Type.OBJECT,
                required=["name"],
                properties={
                    "name":        types.Schema(type=types.Type.STRING),
                    "description": types.Schema(type=types.Type.STRING),
                },
            ),
        ),
        "new_organizations": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(
                type=types.Type.OBJECT,
                required=["name"],
                properties={
                    "name":        types.Schema(type=types.Type.STRING),
                    "description": types.Schema(type=types.Type.STRING),
                },
            ),
        ),
        "new_lore": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(
                type=types.Type.OBJECT,
                required=["fact"],
                properties={
                    "fact": types.Schema(type=types.Type.STRING),
                },
            ),
        ),
    },
)

# ── Extraction functions ─────────────────────────────────────────────────────

def truncate(text: str, max_words: int = 10_000) -> str:
    words = text.split()
    return " ".join(words[:max_words]) + "\n[...truncated...]" if len(words) > max_words else text


def extract_summary(chapter: dict) -> dict:
    prompt = f"""You are a literary analyst. Summarize Chapter {chapter['index']}: "{chapter['title']}".

CHAPTER TEXT:
{truncate(chapter['text'])}

Focus on: what happens, decisions made, conflicts introduced or resolved, and the emotional arc.
Keep the summary between 150-250 words."""
    return call_gemini(prompt, schema=SUMMARY_SCHEMA, max_tokens=4000)


def extract_characters(summary: str, chapter_num: int, reg: KnowledgeRegistry) -> dict:
    prompt = f"""You are extracting character data from Chapter {chapter_num}.

KNOWN CHARACTERS (reference by their exact ID -- only output what is NEW this chapter):
{reg.characters_str()}

CHAPTER SUMMARY:
{summary}

Rules:
- Max 6 characters total.
- For existing characters: is_new=false, use their existing ID, only fill new_traits/new_revelations if genuinely new.
- For new characters: is_new=true, id must be an empty string.
- Omit characters who appear briefly with nothing new to report."""
    return call_gemini(prompt, schema=CHARACTERS_SCHEMA, max_tokens=3500)


def extract_relationships(summary: str, chapter_num: int, reg: KnowledgeRegistry) -> dict:
    if not reg.characters:
        return {"relationships": []}

    char_list = "\n".join(f"  [{cid}] {c['name']}" for cid, c in reg.characters.items())

    prompt = f"""You are extracting relationship changes from Chapter {chapter_num}.

VALID CHARACTER IDs (use ONLY these -- do not invent IDs):
{char_list}

CHAPTER SUMMARY:
{summary}

Rules:
- Max 4 relationships.
- Only include relationships meaningfully shown or changed in this chapter.
- Both IDs must come from the list above.
- If no significant relationship dynamics occur, return an empty array."""
    return call_gemini(prompt, schema=RELATIONSHIPS_SCHEMA, max_tokens=4000)


def extract_world(summary: str, chapter_num: int, reg: KnowledgeRegistry) -> dict:
    prompt = f"""You are extracting world-building facts from Chapter {chapter_num}.

ALREADY KNOWN LOCATIONS:
{reg.locations_str()}

CHAPTER SUMMARY:
{summary}

Rules:
- Only extract things NEWLY introduced in this chapter.
- Max: 3 locations, 2 organizations, 4 lore facts.
- If nothing new in a category, return an empty array for it."""
    return call_gemini(prompt, schema=WORLD_SCHEMA, max_tokens=4000)

print("Schemas and extraction functions ready.")

Schemas and extraction functions ready.


In [34]:
def process_chapter(chapter: dict, reg: KnowledgeRegistry) -> dict:
    n, title = chapter["index"], chapter["title"]
    print(f"\n{'='*55}")
    print(f"  Chapter {n}: {title}  ({chapter['word_count']} words)")
    print(f"{'='*55}")

    print("  [1/4] Summary...")
    summary_data = extract_summary(chapter)
    summary_text = summary_data.get("summary", "")

    print("  [2/4] Character deltas...")
    char_data = extract_characters(summary_text, n, reg)
    reg.apply_characters(char_data, n)

    print("  [3/4] Relationship deltas...")
    rel_data = extract_relationships(summary_text, n, reg)
    reg.apply_relationships(rel_data, n)

    print("  [4/4] World facts...")
    world_data = extract_world(summary_text, n, reg)
    reg.apply_world(world_data, n)

    print(f"\n  Registry state: {len(reg.characters)} chars | "
          f"{len(reg.relationships)} rels | {len(reg.locations)} locations")

    return {
        "chapter_number": n,
        "chapter_title":  title,
        "summary":        summary_data,
        "characters":     char_data,
        "relationships":  rel_data,
        "world_facts":    world_data,
    }

In [47]:
chapters[12]

{'index': 13,
 'title': '12: The Carving',
 'text': '12\n\nTHE CARVING\n\nMy life becomes agony.\n\nMy Sigils are attached to the metacarpus in each hand. Mickey removes the old Red Sigils and cultivates new skin and bone over the wounds. Then he sets to installing a stolen subdermal datachip into my frontal lobe. I am told the trauma killed me and they had to restart my heart. I’ve died twice then. They say I was in a coma for two weeks, but to me it was nothing but a dream. I was in the vale with Eo. She kissed me on the forehead and then I woke and felt the stitches and the pain.\n\nI lie in bed as Mickey tests me. He has me move marbles from one container into other containers coded by colors. I do this for what seems a lifetime.\n\n“We are forming synapses, my darling.”\n\nHe tests me with word puzzles and tries to make me read, but I don’t know how to read. “You will have to learn that for the Institute,” he giggles.\n\nMy dreams are cruel things to wake from. In them, Eo comfort

In [48]:
extract_summary(chapters[12])

    tokens -- input: 5426, output: 340, limit: 4000


{'chapter_number': 13,
 'chapter_title': 'The Carving',
 'summary': 'Chapter 13, "The Carving," details Darrow\'s agonizing transformation into a Gold. After dying twice and waking from a coma, Mickey, the Carver, begins the brutal process. Darrow\'s old Sigils are removed, a datachip is installed in his brain, and his entire skeleton is replaced and strengthened to be six times denser than natural bone. His skin, muscles, and tendons are also altered, causing unimaginable pain. Mickey also enhances Darrow\'s intellect, force-feeding him thousands of years of literature and history. Harmony introduces intense physical training using concentraction machines, pushing Darrow to his physical limits, leading to a conflict with Mickey who fears Darrow\'s body cannot withstand the strain. Darrow, however, embraces the pain, driven by his rage and mission. His body rapidly transforms, becoming incredibly strong and agile, far surpassing even Harmony\'s capabilities. His eyes are replaced with 

In [36]:
registry    = KnowledgeRegistry()
all_results = []

result = process_chapter(chapters[2], registry)
all_results.append(result)

print("\n\n=== RAW EXTRACTION OUTPUT ===")
print(json.dumps(result, indent=2))


  Chapter 3: 2: The Township  (3417 words)
  [1/4] Summary...
  [2/4] Character deltas...
    + New character: [C1] Darrow
    + New character: [C2] Ugly Dan
    + New character: [C3] Dago
    + New character: [C4] Eo
  [3/4] Relationship deltas...
    ~ Relationship: C1 <-> C2 (antagonistic)
    ~ Relationship: C1 <-> C3 (rivalry)
    ~ Relationship: C1 <-> C4 (romantic)
  [4/4] World facts...
    + New location: [L1] The Surface
    + New location: [L2] Darrow's Home
    + New organization: [O1] Red Helldivers
    + New organization: [O2] Gamma Helldivers

  Registry state: 4 chars | 3 rels | 2 locations


=== RAW EXTRACTION OUTPUT ===
{
  "chapter_number": 3,
  "chapter_title": "2: The Township",
  "summary": {
    "chapter_number": 3,
    "chapter_title": "2: The Township",
    "summary": "Darrow, a Red Helldiver, narrowly escapes a deadly mining accident, cutting his foot free and burning his hand while extracting valuable helium-3. Despite the danger, he feels a manic triumph, d

In [37]:
print("=== CHARACTERS ===")
for cid, c in registry.characters.items():
    print(f"\n[{cid}] {c['name']}")
    print(f"  Brief    : {c.get('brief')}")
    print(f"  Traits   : {c.get('traits')}")
    print(f"  Chapter 1: {c.get('revelations', {}).get(1)}")

print("\n=== RELATIONSHIPS ===")
for key, rel in registry.relationships.items():
    a = registry.characters.get(rel["character_a_id"], {}).get("name", "?")
    b = registry.characters.get(rel["character_b_id"], {}).get("name", "?")
    print(f"  {a} <-> {b}: {rel['type']}")
    for ch, dev in rel["developments"].items():
        print(f"    Ch{ch}: {dev}")

print("\n=== LOCATIONS ===")
for lid, loc in registry.locations.items():
    print(f"  [{lid}] {loc['name']}: {loc['description']}")

=== CHARACTERS ===

[C1] Darrow
  Brief    : A Red Helldiver and the protagonist.
  Traits   : ['Manic triumph, defiant, competitive, reflective, deeply connected to Eo.']
  Chapter 1: None

[C2] Ugly Dan
  Brief    : A Gray captain.
  Traits   : ['Taunting, holds a position of authority.']
  Chapter 1: None

[C3] Dago
  Brief    : A rival Gamma Helldiver.
  Traits   : ['Competitive.']
  Chapter 1: None

[C4] Eo
  Brief    : Darrow's wife/partner.
  Traits   : ['Loving, caring, playful, resourceful (acquires coffee).']
  Chapter 1: None

=== RELATIONSHIPS ===
  Darrow <-> Ugly Dan: antagonistic
    Ch3: Darrow endures taunts from Ugly Dan, a Gray captain, and mockingly bows, highlighting their antagonistic dynamic within the social hierarchy.
  Darrow <-> Dago: rivalry
    Ch3: Darrow and Dago engage in a competitive verbal exchange, reinforcing their rivalry as Helldivers vying for the Laurel.
  Darrow <-> Eo: romantic
    Ch3: Darrow and Eo share an intimate moment, with Eo tending h

In [38]:
registry    = KnowledgeRegistry()
all_results = []

for chapter in chapters[:5]:
    try:
        result = process_chapter(chapter, registry)
        all_results.append(result)
    except Exception as e:
        print(f"  ERROR on chapter {chapter['index']}: {e}")
        break

print(f"\n\nSummary: {len(all_results)} chapters processed")
print(f"  Characters:    {len(registry.characters)}")
print(f"  Relationships: {len(registry.relationships)}")
print(f"  Locations:     {len(registry.locations)}")


  Chapter 1: Prologue  (269 words)
  [1/4] Summary...
  [2/4] Character deltas...
    + New character: [C1] The Narrator
    + New character: [C2] The Golden Speaker
    + New character: [C3] Golden Students
  [3/4] Relationship deltas...
    ~ Relationship: C1 <-> C2 (antagonistic)
    ~ Relationship: C1 <-> C3 (antagonistic)
    ~ Relationship: C2 <-> C3 (authoritative_mentor)
  [4/4] World facts...
    + New location: [L1] Golden Academy
    + New organization: [O1] Gold Caste
    + New organization: [O2] Golden Regime

  Registry state: 3 chars | 3 rels | 1 locations

  Chapter 2: 1: Helldiver  (2260 words)
  [1/4] Summary...
  [2/4] Character deltas...
    + New character: [C4] Darrow
    + New character: [C5] Eo
    + New character: [C6] Narol
  [3/4] Relationship deltas...
    ~ Relationship: C4 <-> C5 (Husband-Wife)
    ~ Relationship: C4 <-> C6 (Uncle-Nephew)
  [4/4] World facts...
    + New location: [L2] Mars
    + New location: [L3] helium-3 mines
    + New location: [L4] 

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Run all summaries in parallel
print("Pre-computing summaries for all chapters...")
summaries = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(extract_summary, ch): ch for ch in chapters}
    for future in as_completed(futures):
        ch = futures[future]
        summaries[ch["index"]] = future.result()
        print(f"  ✓ Summary done: {ch['title']}")

print(f"All {len(summaries)} summaries ready.")

In [51]:
MIN_WORDS_FOR_EXTRACTION = 300

valid_chapters = [ch for ch in chapters if ch["word_count"] >= MIN_WORDS_FOR_EXTRACTION]
skipped = [ch for ch in chapters if ch["word_count"] < MIN_WORDS_FOR_EXTRACTION]

print(f"Chapters to process : {len(valid_chapters)}")
print(f"Skipped (too short) : {len(skipped)}")
for ch in skipped:
    print(f"  skipped: '{ch['title']}' ({ch['word_count']} words)")

print("\nPre-computing summaries...")

Chapters to process : 46
Skipped (too short) : 4
  skipped: 'Prologue' (269 words)
  skipped: 'Section 46' (151 words)
  skipped: 'Excerpt from Golden Son' (43 words)
  skipped: 'What’s next onyour reading list?' (24 words)

Pre-computing summaries...


In [52]:
import asyncio

async def call_gemini_async(prompt: str, schema: types.Schema, max_tokens: int = 4096, retries: int = 3) -> dict:
    for attempt in range(retries):
        response = await async_client.aio.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=schema,
                temperature=0.1,
                max_output_tokens=max_tokens,
            ),
        )

        usage = response.usage_metadata
        print(f"    tokens -- input: {usage.prompt_token_count}, output: {usage.candidates_token_count}")

        if response.text is None:
            finish_reason = response.candidates[0].finish_reason if response.candidates else "unknown"
            print(f"    attempt {attempt + 1}/{retries} failed -- finish_reason: {finish_reason}")
            if attempt < retries - 1:
                await asyncio.sleep(2 ** attempt)
                continue
            raise ValueError(f"No text after {retries} attempts. finish_reason: {finish_reason}")

        try:
            return json.loads(response.text)
        except json.JSONDecodeError:
            print(f"RAW RESPONSE (failed to parse):\n{response.text[:800]}")
            raise


async def extract_summary_async(chapter: dict, semaphore: asyncio.Semaphore) -> tuple[int, dict]:
    async with semaphore:  # semaphore limits how many run at once
        word_count = chapter["word_count"]
        target_words = min(300, max(150, word_count // 15))
        prompt = f"""You are a literary analyst. Summarize Chapter {chapter['index']}: "{chapter['title']}".

CHAPTER TEXT:
{truncate(chapter['text'])}

Focus on: what happens, decisions made, conflicts introduced or resolved, and the emotional arc.
Keep the summary to around {target_words} words."""
        result = await call_gemini_async(prompt, schema=SUMMARY_SCHEMA)
        return chapter["index"], result


async def precompute_summaries(chapters: list[dict], max_concurrent: int = 4) -> dict:
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [extract_summary_async(ch, semaphore) for ch in chapters]
    
    summaries = {}
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    for ch, result in zip(chapters, results):
        if isinstance(result, Exception):
            print(f"  ✗ {ch['title']} FAILED: {result}")
        else:
            idx, data = result
            summaries[idx] = data
            print(f"  ✓ {ch['title']}")
    
    return summaries


async_client = genai.Client(api_key=GEMINI_API_KEY)

In [53]:
print("Pre-computing summaries...")
summaries = await precompute_summaries(valid_chapters, max_concurrent=4)
print(f"\n{len(summaries)}/{len(valid_chapters)} summaries ready.")

Pre-computing summaries...
    tokens -- input: 3791, output: 274
    tokens -- input: 3047, output: 262
    tokens -- input: 4583, output: 332
    tokens -- input: 3749, output: 291
    tokens -- input: 3533, output: 268
    tokens -- input: 3133, output: 258
    tokens -- input: 1862, output: 216
    tokens -- input: 1723, output: 223
    tokens -- input: 3984, output: 270
    tokens -- input: 4071, output: 308
    tokens -- input: 1380, output: 243
    tokens -- input: 5423, output: 361
    tokens -- input: 3256, output: 213
    tokens -- input: 1818, output: 242
    tokens -- input: 2756, output: 226
    tokens -- input: 2300, output: None
    attempt 1/3 failed -- finish_reason: unknown
    tokens -- input: 2300, output: None
    attempt 2/3 failed -- finish_reason: unknown
    tokens -- input: 1864, output: 206
    tokens -- input: 5148, output: 250
    tokens -- input: 2300, output: None
    attempt 3/3 failed -- finish_reason: unknown
    tokens -- input: 2367, output: 237
    

In [ ]:
def process_chapter(chapter: dict, reg: KnowledgeRegistry, precomputed_summary: dict = None) -> dict:
    n, title = chapter["index"], chapter["title"]
    print(f"\n{'='*55}")
    print(f"  Chapter {n}: {title}  ({chapter['word_count']} words)")
    print(f"{'='*55}")

    # Use pre-computed summary if available, otherwise compute now
    print("  [1/4] Summary...")
    summary_data = precomputed_summary or extract_summary(chapter)
    summary_text = summary_data.get("summary", "")

    # Characters + world facts in parallel (both only need the summary)
    print("  [2+4] Characters + World facts (parallel)...")
    with ThreadPoolExecutor(max_workers=2) as executor:
        char_future  = executor.submit(extract_characters, summary_text, n, reg)
        world_future = executor.submit(extract_world, summary_text, n, reg)
        char_data  = char_future.result()
        world_data = world_future.result()

    reg.apply_characters(char_data, n)

    # Relationships must come after characters are registered
    print("  [3/4] Relationships...")
    rel_data = extract_relationships(summary_text, n, reg)
    reg.apply_relationships(rel_data, n)
    reg.apply_world(world_data, n)

    print(f"\n  Registry: {len(reg.characters)} chars | {len(reg.relationships)} rels | {len(reg.locations)} locs")

    return {
        "chapter_number": n,
        "chapter_title":  title,
        "summary":        summary_data,
        "characters":     char_data,
        "relationships":  rel_data,
        "world_facts":    world_data,
    }

In [ ]:
registry    = KnowledgeRegistry()
all_results = []

for chapter in chapters:
    try:
        result = process_chapter(
            chapter,
            registry,
            precomputed_summary=summaries.get(chapter["index"])
        )
        all_results.append(result)
    except Exception as e:
        print(f"  ERROR on chapter {chapter['index']}: {e}")
        break

print(f"\nDone: {len(all_results)} chapters processed")
print(f"  Characters: {len(registry.characters)} | Relationships: {len(registry.relationships)} | Locations: {len(registry.locations)}")